# Reproducing a debtrank-globe scenario

Everything the [live application](https://lastch1ld.github.io/debtrank-globe/)
shows is reproducible from two public pieces: the `debtrank-model` package and
the per-year network snapshots served from GitHub Pages.

This notebook takes one scenario end to end — fetch a year, build the network,
run DebtRank, read the ranking — and then shows the two things that change the
answer most: the portfolio-investment layer, and shocking more than one country.

```bash
pip install debtrank-model requests
```


In [ ]:
import requests
from debtrank_model import build_exposure_network, run_debtrank, clearing_vector

SNAPSHOT_URL = "https://lastch1ld.github.io/debtrank-globe/data/network/{year}.json"

YEAR = 2020
snapshot = requests.get(SNAPSHOT_URL.format(year=YEAR), timeout=30).json()

len(snapshot["nodes"]), len(snapshot["edges"]), len(snapshot["portfolio_edges"])

## Build the network

`build_exposure_network` is the part that decides what a DebtRank number means.
It resolves each country's loss-absorbing buffer through a fallback chain —
reserves, then 1% of GDP, then the country's own bank-capital ratio applied to
its gross cross-border footprint, then a floor — because reserves and GDP are
meaningless proxies for cross-border financial centres, where BIS counts every
resident bank and reported claims run to multiples of local GDP.

Two people who build their networks differently are not running the same model,
which is why this ships with the algorithm instead of being left to the caller.


In [ ]:
network = build_exposure_network(snapshot)
network.n, network.exposure.shape, network.equity.min()

## Run a shock

A shock is a mapping of country code to initial distress in `[0, 1]`. Distress
propagates along the exposure edges, scaled by how much of the creditor's equity
each exposure represents, until no node's distress is still rising.

This is the 2010 Greek debt crisis preset from the live app, on 2020 data.


In [ ]:
result = run_debtrank(network, {"GRC": 1.0})

def top(result, n=10):
    rows = sorted(zip(result.node_ids, result.final_distress), key=lambda r: -r[1])
    return [(node, round(float(d), 4)) for node, d in rows[:n] if d > 1e-6]

print(f"Aggregate DebtRank impact: {result.debtrank:.4f}")
for node, distress in top(result):
    print(f"  {node}: {distress}")

## The portfolio layer changes the answer

`edges` are BIS bank-to-bank claims. `portfolio_edges` are IMF CPIS cross-border
bond and equity holdings — a largely separate channel, and the one that actually
carried the 2010 euro sovereign crisis. The live app exposes this as "Include
portfolio investment"; here it is one keyword argument.

CPIS is a voluntary survey and its coverage ends at 2023, so 2024 and 2025 carry
an empty `portfolio_edges` array rather than stale numbers.


In [ ]:
with_portfolio = build_exposure_network(snapshot, include_portfolio=True)
result_pf = run_debtrank(with_portfolio, {"GRC": 1.0})

print(f"bank edges only:     {result.debtrank:.4f}")
print(f"+ portfolio layer:   {result_pf.debtrank:.4f}")
print()
for node, distress in top(result_pf):
    print(f"  {node}: {distress}")

## More than one country at a time

The engine takes any number of initially-distressed nodes, which is what makes
it possible to replay a crisis as it actually unfolded rather than as a single
clean default. Magnitudes below are illustrative round numbers, not calibrated
to realised losses.


In [ ]:
cascade = run_debtrank(network, {"GRC": 1.0, "PRT": 0.6, "IRL": 0.5})

print(f"Aggregate DebtRank impact: {cascade.debtrank:.4f}")
for node, distress in top(cascade):
    print(f"  {node}: {distress}")

## Eisenberg–Noe, for comparison

A different question: rather than propagating a distress fraction, solve for the
clearing payment vector and read off who cannot meet their obligations in full.

Cross-border bank liabilities routinely dwarf a country's reserves, so 56 of the
217 countries are already "short" with no shock at all. That is a data-scale
mismatch, not contagion — so the shocked run has to be netted against the
unshocked baseline, exactly as DebtRank nets against its initial state.

The two models disagree, and that is the point of shipping both. Netted this way
Eisenberg–Noe registers **nothing at all** for a full Greek default — its
obligations are simply too small to break anyone's clearing position — while a
US or UK shock does move it. DebtRank, which propagates fractional distress
rather than payment failure, ranks Greece's creditors immediately. Neither is
wrong; they measure different things.


In [ ]:
import numpy as np

def shortfall_ratio(net, shocked=None, magnitude=0.0):
    liabilities = net.exposure.T  # a owes b what b holds a claim on
    external = net.equity.copy()
    if shocked is not None:
        external[net.node_ids.index(shocked)] *= 1 - magnitude
    cv = clearing_vector(net.node_ids, liabilities, external)
    p_bar = np.asarray(cv.nominal_liabilities, dtype=float)
    payments = np.asarray(cv.payments, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(p_bar > 0, np.maximum(0.0, (p_bar - payments) / p_bar), 0.0)

baseline = shortfall_ratio(network)
print(f"already short before any shock: {(baseline > 1e-6).sum()} of {network.n}")

for country in ["GRC", "USA"]:
    marginal = np.maximum(0.0, shortfall_ratio(network, country, 1.0) - baseline)
    ranked = [(network.node_ids[i], round(float(marginal[i]), 4))
              for i in np.argsort(-marginal)[:5] if marginal[i] > 1e-6]
    print()
    print(f"{country} defaults in full -> {ranked or 'no marginal shortfall anywhere'}")

---

Every scenario in the live app is a URL, so any result above can be opened
there directly:

<https://lastch1ld.github.io/debtrank-globe/?year=2020&shock=GRC&magnitude=1.00&model=debtrank&portfolio=1>

Data sources and attribution terms: [`docs/data-api.md`](../docs/data-api.md).
